In [1]:
!pip install pandas scikit-learn xgboost

In [2]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline

import numpy as np

In [3]:
print("working ")

working 


importing libraries

In [4]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder

loading dataset

In [5]:
df = pd.read_csv("believability_dataset_1200.csv", encoding='latin1')

In [6]:
print(df.head())
print(df.columns)

          Scenario  Relationship Seriousness        Tone  \
0  Forgot birthday        Friend         Low   Emotional   
1  Forgot birthday        Friend        High  Apologetic   
2      Missed exam      Academic         Low      Formal   
3   Missed meeting  Professional         Low  Apologetic   
4   Cancelled date      Romantic      Medium   Emotional   

                                         Excuse_Text  Believability_Score  
0  To be honest, I was feeling really overwhelmed...                    1  
1  I know this is no excuse, but I've been dealin...                    4  
2  I received a last-minute call from my boss ask...                    3  
3  I was waiting for a crucial package to arrive ...                    3  
4  I had an unexpected appointment that I complet...                    3  
Index(['Scenario', 'Relationship', 'Seriousness', 'Tone', 'Excuse_Text',
       'Believability_Score'],
      dtype='object')


In [7]:
df = df.rename(columns={
    "Scenario": "scenario",
    "Seriousness": "seriousness",
    "Tone": "tone",
    "Relationship": "relationship",
    "Excuse_Text": "excuse",
    "Believability_Score": "score"
})

In [8]:
X = df.drop("score", axis=1)
y = df["score"]

In [9]:
def boost_cat(X):
    return X * 3   # increase weight (you can try 2–5)

preprocessor = ColumnTransformer(
    transformers=[
        ("scenario_text", TfidfVectorizer(max_features=100), "scenario"),
        ("excuse_text", TfidfVectorizer(max_features=300), "excuse"),
        ("cat", Pipeline([
            ("onehot", OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
            ("boost", FunctionTransformer(boost_cat))
        ]), ["seriousness", "tone", "relationship"])
    ]
)


# preprocessor = ColumnTransformer(
#     transformers=[
#         ("scenario_text", TfidfVectorizer(max_features=100), "scenario"),
#         ("excuse_text", TfidfVectorizer(max_features=300), "excuse"),
#         ("cat", OneHotEncoder(handle_unknown='ignore', sparse_output=False), ["seriousness", "tone", "relationship"])
#     ]
# )

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [11]:
X_train_transformed = preprocessor.fit_transform(X_train)

print("Transformed shape:", X_train_transformed.shape)

Transformed shape: (962, 331)


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

In [13]:
model1 = Pipeline([
    ("preprocess", preprocessor),
    ("regressor", LinearRegression())
])

model2 = Pipeline([
    ("preprocess", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=100, random_state=42))
])

model3 = Pipeline([
    ("preprocess", preprocessor),
    ("regressor", XGBRegressor(n_estimators=100, learning_rate=0.1))
])

In [14]:
model1.fit(X_train, y_train)
model2.fit(X_train, y_train)
model3.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('scenario_text',
                                                  TfidfVectorizer(max_features=100),
                                                  'scenario'),
                                                 ('excuse_text',
                                                  TfidfVectorizer(max_features=300),
                                                  'excuse'),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False)),
                                                                  ('boost',
                                                                   FunctionTransformer(func=<function boost_cat at 0x7d5150ff2a20>))])...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.1,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=None, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=100, n_jobs=None,
                              num_parallel_tree=None, ...))])

In [15]:
pred1 = model1.predict(X_test)
pred2 = model2.predict(X_test)
pred3 = model3.predict(X_test)

mae1 = mean_absolute_error(y_test, pred1)
mae2 = mean_absolute_error(y_test, pred2)
mae3 = mean_absolute_error(y_test, pred3)

print("Linear Regression MAE:", mae1)
print("Random Forest MAE:", mae2)
print("XGBoost MAE:", mae3)

Linear Regression MAE: 0.8712416611875892
Random Forest MAE: 0.7665560165975104
XGBoost MAE: 0.7694177627563477


In [16]:
def is_weak_excuse(excuse):
    weak_words = ["forgot", "no time", "busy", "overslept", "dog ate"]

    for word in weak_words:
        if word in excuse.lower():
            return True
    return False

In [17]:
def is_strong_excuse(excuse):
    strong_words = ["hospital", "medical", "emergency", "accident", "surgery"]

    for word in strong_words:
        if word in excuse.lower():
            return True
    return False

Prediction function

In [18]:
import pandas as pd

def predict_believability(sample_input):
    sample_df = pd.DataFrame([sample_input])

    p1 = model1.predict(sample_df)[0]
    p2 = model2.predict(sample_df)[0]
    p3 = model3.predict(sample_df)[0]
    # Apply penalty for weak excuses in high seriousness
    penalty = 0

    if sample_input["seriousness"] == "high" and is_weak_excuse(sample_input["excuse"]):
      penalty = -1.5

    p1 += penalty
    p2 += penalty
    p3 += penalty

    # Boost for strong excuses in high seriousness
    boost = 0

    if sample_input["seriousness"] == "high" and is_strong_excuse(sample_input["excuse"]):
      boost = +1.2

    p1 += boost
    p2 += boost
    p3 += boost

    # Clip scores between 1 and 5
    p1 = max(1, min(5, p1))
    p2 = max(1, min(5, p2))
    p3 = max(1, min(5, p3))

    print("algo1 (Linear):", round(p1,2))
    print("algo2 (RandomForest):", round(p2,2))
    print("algo3 (XGBoost):", round(p3,2))

    # Dynamic selection logic
    preds = {
        "Linear Regression": p1,
        "Random Forest": p2,
        "XGBoost": p3
    }

    # Choose model whose prediction is closest to median
    values = list(preds.values())
    median_val = sorted(values)[1]

    best_model = min(preds, key=lambda k: abs(preds[k] - median_val))

    print("\nBest performing model:", best_model)
    print("Believability score:", round(preds[best_model],2))

In [19]:
def take_user_input():
    scenario = input("Enter scenario: ")
    seriousness = input("Enter seriousness (low/medium/high): ")
    tone = input("Enter tone (casual/apologetic/formal): ")
    relationship = input("Enter relationship (friend/teacher/colleague/etc): ")
    excuse = input("Enter your excuse: ")

    sample = {
        "scenario": scenario,
        "seriousness": seriousness.lower(),
        "tone": tone.lower(),
        "relationship": relationship.lower(),
        "excuse": excuse
    }

    return sample

In [20]:
user_sample = take_user_input()
predict_believability(user_sample)

Enter scenario: i forgot my assignment
Enter seriousness (low/medium/high): high
Enter tone (casual/apologetic/formal): apologetic
Enter relationship (friend/teacher/colleague/etc): teacher
Enter your excuse: my dog ate my assignment
algo1 (Linear): 1
algo2 (RandomForest): 1.95
algo3 (XGBoost): 1.35

Best performing model: XGBoost
Believability score: 1.35
